# Duke CYBERSEC 520 — AI/ML for Cybersecurity

**Portfolio Edition — Patrick Markey**

This notebook demonstrates an end-to-end machine-learning workflow for network-traffic classification using the CICIDS dataset. The project covers data cleaning, binary attack labeling, balanced sampling, train/test splitting, a Random Forest baseline, and hyperparameter experimentation.

**Portfolio highlights:**
- Built a balanced 2,000-row working sample from benign and web-attack traffic.
- Trained a Random Forest classifier on 78 numerical network-flow features.
- Evaluated models with both accuracy and ATTACK-class F1 score.
- Compared a default baseline against five hyperparameter configurations.
- Preserved the original executed outputs for reproducibility and review.

> Dataset files are not included in this repository. Place the required CICIDS CSVs in the `data/` directory before rerunning the notebook.

---


# Assignment 1: ML Basics

**Name:** [Patrick Markey]  
**Team Members (if any):** [Add names here]

---

## Instructions
1. **Save your work:** In Colab, use **File → Save a copy in Drive**, or download the notebook to your local machine.
2. **Run in order:** Run cells from top to bottom because execution order matters.
3. **Complete the student areas:** Follow the comments in each section.
4. **Required experiments:** Document one baseline model plus at least five different hyperparameter configurations (at least six model runs total).
5. **Required metrics:** Report accuracy and the F1 score for the `ATTACK` class for every run. Use Attack F1 as the primary metric when selecting the best configuration.

---

***Attribution**: This notebook was created by Michael Roman for use in the CYBERSEC 520 course at Duke University. It was adapted from lecture notes. Gemini 3 Pro was used to format and clean up this notebook.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mproman/CYBERSEC520/blob/main/content/assignments/Assignment_1.ipynb)

## Step 0: Notebook Setup
We need to bring in our tools. These are the standard libraries used by Data Scientists everywhere.
* **Pandas:** For handling data like Excel sheets.
* **Seaborn/Matplotlib:** For drawing graphs.
* **Sklearn:** The Machine Learning library.

**Action:** Click the "Play" button on the cell below.

In [205]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine-learning tools
# This will allow us to split our data early on, so we can have a training model and also a testing 
from sklearn.model_selection import train_test_split
# Random Forest Model(I chose this because after researching different training models this model stoodout. 
# I like that it creates many slightly decision trees, and combines those votes. An analogy to this model: Think of it asa team of SOC analysts investigating the same network flow: The problem is one tree might
# focus heaviley on particular examples and make unreliable desicions(Overfitting) ) A Random Forest creates many TREEs instead of just one, it's a bunch of trees 
from sklearn.ensemble import RandomForestClassifier
# Importing the KNeighbors traing model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn import datasets
from sklearn.svm import SVC

# Make the graphs look nice
plt.style.use("seaborn-v0_8-whitegrid")
print("Setup complete! You are ready to start.")

Setup complete! You are ready to start.


## Step 1: Testing with Penguins
Before we tackle a Cyber dataset, let's make sure everything works with a simple dataset: **The Palmer Penguins**.

**Action:** Run the cell below to load the data.

In [206]:
# Load the data from a URL
url = "https://github.com/allisonhorst/palmerpenguins/raw/5b5891f01b52ae26ad8cb9755ec93672f49328a8/data/penguins_size.csv"
penguins = pd.read_csv(url)

# Drop missing values to prevent errors
penguins = penguins.dropna()

# Use two numerical features to predict the species
X_warmup = penguins[["culmen_length_mm", "flipper_length_mm"]]
y_warmup = penguins["species_short"]

# Split into training and test sets while preserving class proportions
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_warmup,
    y_warmup,
    test_size=0.2,
    random_state=42,
    stratify=y_warmup,
)

print("Penguin data loaded and split successfully.")
X_warmup.head()

Penguin data loaded and split successfully.


,culmen_length_mm,flipper_length_mm
0,39.1,181.0
1,39.5,186.0
2,40.3,195.0
4,36.7,193.0
5,39.3,190.0


### 1.1 Train a Simple Model
Now train either a **K-Nearest Neighbors (k-NN)** or **Support Vector Machine Classifier (SVC)** model on the warm-up data.

Use this section to confirm that you can create a model, fit it, generate predictions, and calculate a score before moving to the cybersecurity data.

In [207]:
# 1. Create the model This creates a K Nearest Neighbor Model, but it has not been trained yet. 
# N_Neighbors=5 is a hyperparameter, which is telling the model to examine the five most similar historical netwrok flows when classifying a new flow
model = KNeighborsClassifier(n_neighbors=5)
# 2. Train the model (The "Learning" part) # x is network flow features, y is the answer(BENIGN, or ATTACK) The model can see train during training. 
model.fit(X_train_w, y_train_w)
# 3. Predict on the test set
predictions = model.predict(X_test_w)
print(predictions)
# 4. Check the score
score = accuracy_score(y_test_w, predictions)
print(f"Warmup Accuracy: {score:.2f} (or {score*100:.1f}%)")


['Gentoo' 'Chinstrap' 'Adelie' 'Gentoo' 'Gentoo' 'Gentoo' 'Chinstrap'
 'Adelie' 'Adelie' 'Gentoo' 'Chinstrap' 'Adelie' 'Adelie' 'Chinstrap'
 'Chinstrap' 'Adelie' 'Chinstrap' 'Adelie' 'Chinstrap' 'Gentoo'
 'Chinstrap' 'Gentoo' 'Adelie' 'Gentoo' 'Gentoo' 'Adelie' 'Adelie'
 'Chinstrap' 'Gentoo' 'Chinstrap' 'Gentoo' 'Adelie' 'Adelie' 'Adelie'
 'Adelie' 'Adelie' 'Gentoo' 'Gentoo' 'Adelie' 'Adelie' 'Adelie' 'Gentoo'
 'Chinstrap' 'Gentoo' 'Chinstrap' 'Adelie' 'Gentoo' 'Chinstrap' 'Gentoo'
 'Chinstrap' 'Gentoo' 'Adelie' 'Chinstrap' 'Adelie' 'Gentoo' 'Adelie'
 'Adelie' 'Chinstrap' 'Adelie' 'Gentoo' 'Adelie' 'Gentoo' 'Gentoo'
 'Adelie' 'Gentoo' 'Adelie' 'Adelie']
Warmup Accuracy: 0.97 (or 97.0%)


## Step 2: Cybersecurity Data
Now that the warm-up works, use the assignment data.

**Instructions:**
1. Download `Monday-WorkingHours.pcap_ISCX.csv` and one attack-day CSV from the [Canvas Datasets folder](https://canvas.duke.edu/courses/82545/files/folder/Datasets).
2. Combine the Monday file with the attack-day file. **Do not use Monday by itself:** it contains only benign traffic.
3. Create a binary target with two classes: `BENIGN` and `ATTACK`.
4. Clean missing, infinite, and invalid values before modeling.
5. Subsample the data to reduce runtime.
6. Update both path variables in the loading cell below.

> **Sampling hint:** A simple random sample may omit rare attacks. Consider sampling from `BENIGN` and `ATTACK` separately. When splitting the sample, use a fixed random seed and `stratify=y` so both classes appear in the training and test sets.

The Google Drive cell is optional and is needed only when your files are stored in Drive while using Colab.

In [208]:
# Optional: mount Google Drive when running this notebook in Colab.
# Skip this cell if you uploaded files directly, use Kaggle, or run locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Drive mounting is only needed in Colab.")

Google Drive mounting is only needed in Colab.


In [209]:
# Portable repository-relative dataset paths.
from pathlib import Path

DATA_DIR = Path("data")
benign_path = DATA_DIR / "Monday-WorkingHours.pcap_ISCX.csv"
attack_path = DATA_DIR / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"

# Confirm the required files exist before loading them.
missing = [str(p) for p in (benign_path, attack_path) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset file(s): " + ", ".join(missing) +
        ". Place the CICIDS CSV files in the repository's data/ folder."
    )

# Load and combine the benign and attack-day datasets.
df_benign = pd.read_csv(benign_path)
df_attack = pd.read_csv(attack_path)
df = pd.concat([df_benign, df_attack], ignore_index=True)

print(f"Combined size: {len(df):,} rows")


Combined size: 700,284 rows


### 2.1 Clean, Label, and Subsample
Cybersecurity datasets are large and often contain missing or infinite values.

In the next cell:

1. Normalize the column names.
2. Remove invalid values.
3. Convert the original labels into binary `BENIGN` and `ATTACK` labels.
4. Create a smaller sample for faster experiments.
5. Print the class counts and verify that both classes are present.

**Hint:** Sample from each class separately instead of taking one unrestricted random sample. The comments provide guidance, but you must complete the implementation.

In [210]:
# Complete each step below.

# 1. Normalize column names so " Label" and "Label" are handled consistently.
# df.columns = ...

# 2. Replace positive/negative infinity with NaN, then remove invalid rows.
# df.replace(...)
# df = ...

# 3. Create a binary target named "BinaryLabel".
# Keep BENIGN as "BENIGN" and map every other original label to "ATTACK".
# df["BinaryLabel"] = ...

# 4. Subsample the data.
# HINT: A single random sample may omit rare attacks. Consider sampling
# separately from each BinaryLabel group, using the same random_state.
#
# One possible structure is:
# pieces = []
# for label, group in df.groupby("BinaryLabel"):
#     pieces.append(group.sample(n=min(1000, len(group)), random_state=42))
# df_small = pd.concat(pieces, ignore_index=True)
# df_small = df_small.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Verify the result before continuing.
# Print the original size, sampled size, and BinaryLabel value counts.
# Confirm that both BENIGN and ATTACK are present.
# 1. Clean up weird values (Infinite numbers and empty spots)
# Save the original size before cleaning
original_size = len(df)

# 1. Normalize column names
# This changes " Label" into "Label"
df.columns = df.columns.str.strip()

# 2. Replace infinity with NaN and remove invalid rows
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna().copy()

# Also remove extra spaces inside the label values
df["Label"] = df["Label"].astype(str).str.strip()

# 3. Create the binary target
# BENIGN remains BENIGN; every attack subtype becomes ATTACK
df["BinaryLabel"] = np.where(
    df["Label"].str.upper() == "BENIGN",
    "BENIGN",
    "ATTACK"
)

# 4. Sample each class separately
pieces = []

for label, group in df.groupby("BinaryLabel"):
    class_sample = group.sample(
        n=min(1000, len(group)),
        random_state=42
    )
    pieces.append(class_sample)

# Combine and shuffle the separately sampled classes
df_small = pd.concat(pieces, ignore_index=True)

df_small = df_small.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# 5. Verify everything
print(f"Original size: {original_size:,} rows")
print(f"Cleaned size: {len(df):,} rows")
print(f"Sampled size: {len(df_small):,} rows")
print()
print(df_small["BinaryLabel"].value_counts())

# Stop the notebook if either class is missing
assert set(df_small["BinaryLabel"].unique()) == {
    "BENIGN",
    "ATTACK"
}

df["BinaryLabel"].value_counts()



Original size: 700,284 rows
Cleaned size: 699,712 rows
Sampled size: 2,000 rows

BinaryLabel
BENIGN    1000
ATTACK    1000
Name: count, dtype: int64


BinaryLabel
BENIGN    697532
ATTACK      2180
Name: count, dtype: int64

### 2.2 Prepare the Data
Separate the **features** from the binary **target**, and then create training and test sets.

Use `BinaryLabel` as the target. Do not include either the original `Label` column or `BinaryLabel` among the model features.

Use the same sampled data, preprocessing, split, and random seed for the baseline and all five hyperparameter configurations.

In [211]:
# Complete each step below.

# 1. Inspect the available columns and class counts.

# 2. Define the target.
# y = df_small["BinaryLabel"]

# 3. Define the feature matrix.
# Drop both the original "Label" column and "BinaryLabel" from X.
# X = ...

# 4. Create one fixed stratified train/test split.
# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y,
# )

# 5. Optional but recommended for k-NN and SVM: scale numerical features.
# Fit StandardScaler only on X_train, then transform X_train and X_test.
# Use exactly the same prepared data for the baseline and all experiments.
# For the dataset in class, the column name for the target is ' Label' (with a space) make sure this maps to your dataset 'Label'
# Let's check the columns first
print(df_small.columns)
print()
print(df_small["BinaryLabel"].value_counts())

# 2. Define the target
y = df_small["BinaryLabel"]

# 3. Define the features
# Both labels must be removed so the model cannot see the answer
X = df_small.drop(columns=["Label", "BinaryLabel"])

print(f"\nNumber of features: {X.shape[1]}")
print(f"Non-numeric features: {X.select_dtypes(exclude=np.number).columns.tolist()}")

# 4. Create one fixed, stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"\nTraining rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

print("\nCyber data is ready for training.")

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

## Step 3: Baseline Model
The baseline is your chosen classifier using its **default hyperparameters**, such as `KNeighborsClassifier()` or `SVC()`.

Keep the dataset, preprocessing, train/test split, and random seed unchanged for every later experiment. Report both accuracy and Attack F1. Use Attack F1 as the primary metric when determining whether a configuration improves upon the baseline.

**Action:** Complete the student area below.

In [212]:
# --- STUDENT AREA ---

# 1. Initialize your chosen model with default hyperparameters.
# Example model classes: KNeighborsClassifier or SVC
# How does the default model perform?
clf_baseline = RandomForestClassifier(
    random_state=42
)
# 2. Fit the model on X_train and y_train.
# ...
clf_baseline.fit(X_train, y_train)
# 3. Predict on X_test.

# y_pred_base = ...
y_pred_base = clf_baseline.predict(X_test)
# --- END STUDENT AREA ---

# Calculate the required baseline metrics.
baseline_accuracy = accuracy_score(y_test, y_pred_base)
baseline_attack_f1 = f1_score(y_test, y_pred_base, pos_label="ATTACK")

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Attack F1: {baseline_attack_f1:.4f}")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_base, digits=4))

print(clf_baseline.get_params()["bootstrap"])
print(clf_baseline.get_params()["n_estimators"])

Baseline accuracy: 0.9925
Baseline Attack F1: 0.9925

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     0.9900    0.9950    0.9925       200
      BENIGN     0.9950    0.9900    0.9925       200

    accuracy                         0.9925       400
   macro avg     0.9925    0.9925    0.9925       400
weighted avg     0.9925    0.9925    0.9925       400

True
100


## Step 4: Hyperparameter Exploration
Train and evaluate at least **five different hyperparameter configurations** in addition to the baseline. Your notebook must therefore contain at least **six model runs total**.

For k-NN, possible hyperparameters include:

- `n_neighbors`
- `weights`
- `metric`

For SVM, possible hyperparameters include:

- `C`
- `kernel`
- `gamma`
- `degree`

The `gamma` parameter applies to the `rbf`, `poly`, and `sigmoid` kernels.

For every configuration:

1. Train the model using the same training data and preprocessing.
2. Predict on the same test data.
3. Record accuracy and Attack F1.
4. Compare the result with the baseline.

Duplicate the experiment cell as needed. Additional experiments are encouraged.

In [213]:
# --- EXPERIMENT TEMPLATE ---
# Duplicate this cell until you have at least five different configurations.
# --- Hyperparamters of the random forest ---
# 1. min_samples_leaf
# 2. n_estimators(I decided to choose this hyperparamter, more trees may improve accuracy, and it did not have excessive computation time)
# 3. max_features
# 4. bootstrap
# 5. max_samples
# 6. max_depth
# --- Advanced Hyperparamter techniques ---
# 1. Grid Search
# 2. Randomized Search
# 3. Bayesian Optimization


experiment_name = "Configuration 1"

# 1. Initialize the same model type with different hyperparameters.
# clf_experiment = ...
clf_experiment = RandomForestClassifier(
    random_state=42,
    n_estimators=50,
    max_features="log2",
    min_samples_split=5,
    max_samples=0.5,
    max_depth=15,
)
# 2. Fit the model using the same training data and preprocessing.
# ...
clf_experiment.fit(X_train, y_train)

# 3. Predict on the same test data.
# y_pred_experiment = ...
y_pred_experiment = clf_experiment.predict(X_test)
# 4. Calculate the required metrics.
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")
# I used AI, to help me create a way to classify the data visually better.
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_experiment, digits=4))

# 5. Record the hyperparameters and metrics in the experiment table.
# 6. Compare this configuration with the default-parameter baseline.
# Configuration 1 missed more attacks then the baseline


Configuration 1
Accuracy: 0.9900
Attack F1: 0.9899

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     1.0000    0.9800    0.9899       200
      BENIGN     0.9804    1.0000    0.9901       200

    accuracy                         0.9900       400
   macro avg     0.9902    0.9900    0.9900       400
weighted avg     0.9902    0.9900    0.9900       400



In [214]:
experiment_name = "Configuration 2"

clf_experiment = RandomForestClassifier(
    random_state=42,
    n_estimators=90,
    max_features="sqrt",
    min_samples_split=10,
    max_samples=1.0,
    max_depth=30,
    min_samples_leaf=5,
)

clf_experiment.fit(X_train, y_train)

y_pred_experiment = clf_experiment.predict(X_test)

experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_experiment, digits=4))


    

Configuration 2
Accuracy: 0.9900
Attack F1: 0.9899

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     0.9949    0.9850    0.9899       200
      BENIGN     0.9851    0.9950    0.9900       200

    accuracy                         0.9900       400
   macro avg     0.9900    0.9900    0.9900       400
weighted avg     0.9900    0.9900    0.9900       400



In [215]:
experiment_name = "Configuration 3"

clf_experiment = RandomForestClassifier(
    random_state=42,
    n_estimators=30,
    bootstrap=False,
    max_features="log2",
    min_samples_leaf=5,
    max_depth=15,
)

clf_experiment.fit(X_train, y_train)

y_pred_experiment = clf_experiment.predict(X_test)
experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_experiment, digits=4))


    


Configuration 3
Accuracy: 0.9900
Attack F1: 0.9900

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     0.9900    0.9900    0.9900       200
      BENIGN     0.9900    0.9900    0.9900       200

    accuracy                         0.9900       400
   macro avg     0.9900    0.9900    0.9900       400
weighted avg     0.9900    0.9900    0.9900       400



In [216]:

experiment_name = "Configuration 4"

clf_experiment = RandomForestClassifier(
    random_state=42,
    n_estimators=5000,
    max_features="log2",
    min_samples_split=10,
    max_samples=1.0,
    max_depth=30,
    min_samples_leaf=5,
)

clf_experiment.fit(X_train, y_train)

y_pred_experiment = clf_experiment.predict(X_test)

experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(y_test, y_pred_experiment, pos_label="ATTACK")

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_experiment, digits=4))

    
    


    

Configuration 4
Accuracy: 0.9850
Attack F1: 0.9848

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     0.9949    0.9750    0.9848       200
      BENIGN     0.9755    0.9950    0.9851       200

    accuracy                         0.9850       400
   macro avg     0.9852    0.9850    0.9850       400
weighted avg     0.9852    0.9850    0.9850       400



In [217]:
experiment_name = "Configuration 5"

clf_experiment = RandomForestClassifier(
    random_state=42,
    n_estimators=300,
)

clf_experiment.fit(X_train, y_train)

y_pred_experiment = clf_experiment.predict(X_test)

experiment_accuracy = accuracy_score(y_test, y_pred_experiment)
experiment_attack_f1 = f1_score(
    y_test,
    y_pred_experiment,
    pos_label="ATTACK"
)

print(experiment_name)
print(f"Accuracy: {experiment_accuracy:.4f}")
print(f"Attack F1: {experiment_attack_f1:.4f}")

print("\nCompared with baseline:")
print(f"Accuracy difference: {experiment_accuracy - baseline_accuracy:+.4f}")
print(f"Attack F1 difference: {experiment_attack_f1 - baseline_attack_f1:+.4f}")

print("\nDetailed classification report:")
print(classification_report(y_test, y_pred_experiment, digits=4))


Configuration 5
Accuracy: 0.9925
Attack F1: 0.9925

Compared with baseline:
Accuracy difference: +0.0000
Attack F1 difference: +0.0000

Detailed classification report:
              precision    recall  f1-score   support

      ATTACK     0.9900    0.9950    0.9925       200
      BENIGN     0.9950    0.9900    0.9925       200

    accuracy                         0.9925       400
   macro avg     0.9925    0.9925    0.9925       400
weighted avg     0.9925    0.9925    0.9925       400



## Step 5: Documentation and Reflection
*Double-click this cell to edit it and answer every question.*

### Experiment summary

| Experiment | Hyperparameters | Accuracy | Attack F1 |
|---|---|---:|---:|
| Baseline | Default settings | | |
| Configuration 1 | 
 random_state=42,
n_estimators=50,
max_features="log2",
min_samples_split=5,
max_samples=0.5,
    max_depth=15, | | |
| Configuration 2 | | | |
| Configuration 3 | | | |
| Configuration 4 | | | |
| Configuration 5 | | | |

### Reflection questions

1. **What dataset and model did you choose?**
2. **How did you combine, clean, and subsample the data?**
3. **How many `BENIGN` and `ATTACK` observations were in your final sample?**
4. **What were the baseline accuracy and Attack F1?**
5. **Which hyperparameters and configurations did you test?**
6. **Which configuration performed best according to Attack F1?**
7. **How much did the best configuration improve over the baseline?**
8. **Why do you think that configuration performed better?**
9. **Did accuracy and Attack F1 lead to the same conclusion? Why or why not?**
10. **What problems or errors did you encounter, and how did you resolve them?**
11. **If you had more time, what would you try next?**

Evidence of thoughtful exploration is where the bulk of the assignment grade comes from.

## Submission Checklist

- [X ] I included my name.
- [X ] I listed all team members, if any.
- [ X] I combined the Monday benign dataset with one attack-day dataset.
- [ ] My final sample contains both `BENIGN` and `ATTACK` observations.
- [ ] I used one fixed train/test split for every experiment.
- [ ] I documented one default-parameter baseline and at least five hyperparameter configurations.
- [ ] I reported accuracy and Attack F1 for every run.
- [ ] I completed every reflection question.
- [ ] I ran all completed cells in order and preserved their outputs.
- [ ] I documented generative-AI use according to the syllabus.

**To submit:** In Colab, select **File → Download → Download .ipynb**. Submit the completed notebook and, if your documentation is separate, the separate write-up through Canvas.